In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [17]:
d = np.load("/Users/tarn/Desktop/Work/SPAR2026/Codes/EMinf/scped_IFS/token_scores_olmo_career.npz")
df = pd.DataFrame({k: d[k] for k in d.files})

di = pd.read_csv("/Users/tarn/Desktop/Work/SPAR2026/Codes/EMinf/scped_IFS/token_scores_olmo_career_top_select_extremes_sidetop.npz")

print(df.head(10))
print(di.head(10))

long_rows = []
for label in "ABCDE":
    sub = di[["example_idx", "correct_label", f"position_{label}", f"token_id_{label}"]].copy()
    sub = sub.rename(columns={f"position_{label}": "position", f"token_id_{label}": "token_id"})
    sub["label"] = label
    sub["is_correct"] = sub["correct_label"] == label
    long_rows.append(sub)
di_long = pd.concat(long_rows, ignore_index=True)

print("di_long: \n", di_long)

merged = di_long.merge(df[["example_idx", "position", "token_id", "score"]], on=["example_idx", "position"])
assert (merged["token_id_x"] == merged["token_id_y"]).all() if "token_id_x" in merged else True  # sanity check, see note below




   example_idx  position  token_id        score
0            0        74      1271  -671.944580
1            0        75      3480   592.391663
2            0        76       505  -129.717651
3            0        77     33400  -240.599060
4            0        78       311     7.384064
5            0        79      3241     0.797302
6            0        80     15009    -1.183472
7            0        81        11   289.085571
8            0        82       499  -546.612427
9            0        83       649  1281.483643
   example_idx side correct_label  position_A  token_id_A  position_B  \
0            0  top             D         128        1455         132   
1            1  top             D          81        6721          88   
2            2  top             A          91       17257         110   
3            3  top             C         114         814         117   
4            4  top             E         117        5603         125   
5            5  top             B 

In [ ]:
results = pd.read_csv("/Users/tarn/Desktop/Work/SPAR2026/Codes/EMinf/token_rub_scores/intruder/top/smoketest_results.csv")
print(f"{len(results)} documents")
print(results.head(10))

# --- sanity checks before trusting this enough to scale up to the full run ---
accuracy = results["is_correct"].mean()
n_unanswered = results["predicted_label"].isna().sum()
print(f"\naccuracy: {accuracy:.3f}  (chance = 0.200, n={len(results)})")
print(f"unanswered (predicted_label is NaN): {n_unanswered}/{len(results)}")

print("\nnumeric_mass distribution (low mass across the board would suggest the judge isn't")
print("reliably returning a clean A-E letter):")
print(results["numeric_mass"].describe())

print("\npredicted_label distribution -- watch for one letter dominating regardless of")
print("correct_label: could be a real positional bias, or a bug where it just defaults to")
print("one answer no matter what the prompt says:")
print(results["predicted_label"].value_counts())

print("\ncorrect_label distribution (should be roughly uniform across A-E, matching what")
print("select_extreme_tokens.py itself already printed when it generated this file):")
print(results["correct_label"].value_counts())

print("\na few full rows to eyeball directly:")
print(results[["example_idx", "correct_label", "predicted_label", "is_correct", "numeric_mass"]].head(10))